# 07 - Exploratory data analysis

This notebook describes the class balance, review length distribution and
domain composition of the training corpus and of the three evaluation
scenarios. Figures are written to `results/figures/`.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from spam_detection.config import (
    CLASSIFIER_TEST_PATH,
    CLASSIFIER_TRAIN_PATH,
    CROSS_DOMAIN_PATH,
    HOTEL_REVIEWS_PATH,
    OPSPAM_PREPROCESSED_PATH,
    RESULTS_DIR,
    YELP_PREPROCESSED_PATH,
)

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 150, "font.family": "serif"})

## Load datasets

In [ ]:
train = pd.read_csv(CLASSIFIER_TRAIN_PATH)
test = pd.read_csv(CLASSIFIER_TEST_PATH)
opspam = pd.read_csv(OPSPAM_PREPROCESSED_PATH)
yelp = pd.read_csv(YELP_PREPROCESSED_PATH)
hotels = pd.read_csv(HOTEL_REVIEWS_PATH)
cross_domain = pd.read_csv(CROSS_DOMAIN_PATH)

datasets = {
    "Train": train,
    "Test": test,
    "OpSpam": opspam,
    "Yelp": yelp,
    "Hotel Reviews": hotels,
    "Cross-Domain": cross_domain,
}

## Class balance

In [ ]:
balance = pd.DataFrame(
    {
        name: frame["label"].value_counts(normalize=True).sort_index()
        for name, frame in datasets.items()
    }
).T
balance.columns = ["Real (0)", "Fake (1)"]
display(balance.round(4))

ax = balance.plot(kind="bar", stacked=True, color=["#4c72b0", "#c44e52"], figsize=(8, 4))
ax.set_ylabel("Share of reviews")
ax.set_title("Class balance by dataset")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_balance.png", bbox_inches="tight")
plt.show()

## Review length distribution

In [ ]:
length_rows = []
for name, frame in datasets.items():
    lengths = frame["text"].fillna("").astype(str).str.split().str.len()
    length_rows.append(
        {
            "Dataset": name,
            "Reviews": len(lengths),
            "Mean words": round(lengths.mean(), 2),
            "Median words": round(lengths.median(), 2),
            "P95 words": round(lengths.quantile(0.95), 2),
            "Max words": int(lengths.max()),
        }
    )
length_table = pd.DataFrame(length_rows)
display(length_table)

fig, ax = plt.subplots(figsize=(8, 4))
for name in ("OpSpam", "Yelp", "Hotel Reviews", "Cross-Domain"):
    lengths = datasets[name]["text"].fillna("").astype(str).str.split().str.len()
    ax.hist(lengths.clip(upper=400), bins=40, alpha=0.5, label=name)
ax.set_xlabel("Words per review (clipped at 400)")
ax.set_ylabel("Frequency")
ax.set_title("Review length distribution")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "review_lengths.png", bbox_inches="tight")
plt.show()

## Source and domain composition

In [ ]:
display(train["source"].value_counts().rename("Train"))
display(train["domain"].value_counts().rename("Train"))
display(cross_domain["category"].value_counts().head(15).rename("Cross-domain"))

## Duplicate and empty-text checks

In [ ]:
checks = []
for name, frame in datasets.items():
    checks.append(
        {
            "Dataset": name,
            "Rows": len(frame),
            "Duplicate texts": int(frame["text"].duplicated().sum()),
            "Empty texts": int(frame["text"].fillna("").astype(str).str.strip().eq("").sum()),
        }
    )
display(pd.DataFrame(checks))